# Training set analysis

Each row is a pair of tetrahedra: 24 coordinates `T{1,2}_v{1-4}_{x,y,z}` plus two targets, `HasIntersection` (0/1) and `IntersectionVolume` (volume of the overlap, 0 when the tetrahedra are disjoint or only touch).

This is the processed training split: the configured `transformations` are applied to the coordinates and `IntersectionVolume` is scaled by `volume_scale_factor`. The scaling is undone below, so every volume here is in original units.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
config = yaml.safe_load(open(ROOT / "config/config.yaml"))["processor_config"]

df = pd.read_csv(ROOT / config["dataset_paths"]["processed_data"] / "train/train_data.csv")
assert df.shape[1] == 26, "expects 24 coordinate columns (no unitary_tetrahedron transformation)"
df["IntersectionVolume"] /= config["volume_scale_factor"]

# (pair, tetrahedron, vertex, axis) and the volume of each tetrahedron: |det(edges from vertex 1)| / 6
tetrahedra = df.iloc[:, :24].to_numpy().reshape(-1, 2, 4, 3)
volumes = np.abs(np.linalg.det(tetrahedra[:, :, 1:] - tetrahedra[:, :, :1])) / 6
overlap = df["IntersectionVolume"].to_numpy()

CASES = ["no intersection", "touching only", "volumetric overlap"]
case = np.select([df["HasIntersection"] == 0, overlap == 0], CASES[:2], CASES[2])

plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})


def note(ax, text):
    ax.text(0.97, 0.97, text, transform=ax.transAxes, va="top", ha="right",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", alpha=0.9))


def log_bins(values, n):
    return np.logspace(np.log10(values.min()), np.log10(values.max()), n)


df.head()

In [ ]:
df.describe()

## Class balance

In [ ]:
counts = pd.Series(case).value_counts().reindex(CASES, fill_value=0)
print(pd.DataFrame({"samples": counts, "percent": (100 * counts / len(df)).round(2)}))
intersecting = counts["touching only"] + counts["volumetric overlap"]
print(f"\nOf the {intersecting:,} intersecting pairs, {counts['volumetric overlap'] / intersecting:.1%} overlap with volume > 0; the rest only touch")

## Coordinates

Raw coordinates are i.i.d. uniform in [0, 1] (mean 0.5, std 0.289); the transformations move them away from that.

In [ ]:
all_coords = tetrahedra.ravel()
print(f"{len(all_coords):,} values in [{all_coords.min():.4f}, {all_coords.max():.4f}], mean {all_coords.mean():.4f}, std {all_coords.std():.4f}")

panels = [
    ("All coordinates", [all_coords], ["all"]),
    ("X vs Y vs Z", [tetrahedra[..., k].ravel() for k in range(3)], ["x", "y", "z"]),
    ("T1 vs T2", [tetrahedra[:, t].ravel() for t in range(2)], ["T1", "T2"]),
]
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (title, data, labels) in zip(axes, panels):
    ax.hist(data, bins=50, label=labels, alpha=0.7)
    ax.set(title=title, xlabel="Coordinate value", ylabel="Count")
    ax.legend()
plt.tight_layout()

## Tetrahedron volumes

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, v, name in zip(axes, volumes.T, ["T1", "T2"]):
    v = v[v > 1e-12]
    ax.hist(v, bins=log_bins(v, 50))
    ax.set(xscale="log", yscale="log", title=f"{name} volume distribution", xlabel="Volume", ylabel="Count")
    note(ax, f"n = {len(v):,}\nmean {v.mean():.2e}\nmedian {np.median(v):.2e}")
plt.tight_layout()

pd.DataFrame(volumes, columns=["T1 volume", "T2 volume"]).groupby(case).agg(["count", "mean", "std"]).reindex(CASES)

In [ ]:
subsets = [(name, case == name) for name in CASES] + [("all data", np.ones(len(df), dtype=bool))]
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
for ax, (name, mask) in zip(axes.flat, subsets):
    v = volumes[mask & (volumes > 1e-12).all(axis=1)]
    ax.set(title=name, xlabel="T1 volume", ylabel="T2 volume")
    if len(v) == 0:
        note(ax, "no data")
        continue
    hb = ax.hexbin(v[:, 0], v[:, 1], gridsize=30, mincnt=1, xscale="log", yscale="log", cmap="viridis")
    fig.colorbar(hb, ax=ax, label="Count")
    ax.plot([v.min(), v.max()], [v.min(), v.max()], "r--", label="T1 = T2")
    ax.legend(loc="lower right")
    note(ax, f"n = {len(v):,}")
plt.tight_layout()

## Volume difference (T1 - T2)

In [ ]:
diff = volumes[:, 0] - volumes[:, 1]
fig, (linear, log) = plt.subplots(1, 2, figsize=(16, 6))

linear.hist(diff, bins=60)
linear.axvline(0, color="black", linestyle="--", label="zero")
linear.axvline(diff.mean(), color="red", label=f"mean ({diff.mean():.4f})")
linear.set(title="Linear scale", xlabel="T1 - T2", ylabel="Count")
linear.legend()

for values, label in ((diff[diff > 0], "T1 > T2"), (-diff[diff < 0], "T2 > T1")):
    if len(values):
        log.hist(values, bins=log_bins(values, 25), alpha=0.7, label=f"{label}: {len(values):,}")
log.set(xscale="log", title="Log scale (absolute differences)", xlabel="|T1 - T2|", ylabel="Count")
log.legend()
plt.tight_layout()

percentiles = [5, 25, 50, 75, 95]
print(pd.Series(np.percentile(diff, percentiles), index=[f"P{p}" for p in percentiles]).to_string(float_format="{:.6f}".format))
print("\nMagnitude |T1 - T2|:")
print(pd.Series(pd.cut(np.abs(diff), [0, 0.01, 0.1, np.inf], right=False)).value_counts(normalize=True).sort_index().round(3).to_string())
_, p_value = stats.ttest_1samp(diff, 0)
print(f"\nT1 larger in {(diff > 0).mean():.1%} of pairs; mean {diff.mean():.6f}, std {diff.std():.6f}, |mean|/std {abs(diff.mean()) / diff.std():.3f}; t-test against 0: p = {p_value:.3g}")

## Intersection volume

Pairs with a positive overlap volume only.

In [ ]:
vol = overlap[overlap > 0]
print(f"{len(vol):,} volumes in [{vol.min():.2e}, {vol.max():.2e}], median {np.median(vol):.2e}, spanning {np.log10(vol.max() / vol.min()):.1f} orders of magnitude")

limit = 1.2 * np.percentile(vol, 90)
fig, (linear, log) = plt.subplots(1, 2, figsize=(15, 6))
linear.hist(vol[vol <= limit], bins=50)
linear.set(title="Linear scale (up to 1.2 x P90)", xlabel="Volume", ylabel="Count")
note(linear, f"{(vol <= limit).mean():.0%} of samples shown")
log.hist(vol, bins=log_bins(vol, 40))
log.set(xscale="log", title="Log scale (full range)", xlabel="Volume", ylabel="Count")
note(log, f"{len(vol):,} samples\nmedian {np.median(vol):.1e}")
plt.tight_layout()

SMALL, LARGE = 1e-4, 1.4e-2
sizes = pd.Series(pd.cut(vol, [0, SMALL, LARGE, np.inf], right=False, labels=[f"small (< {SMALL:g})", f"medium ({SMALL:g} to {LARGE:g})", f"large (>= {LARGE:g})"]))
print(sizes.value_counts(normalize=True).sort_index().round(3).to_string())